# TensorFlow による MNIST 画像分類

このノートブックでは、Keras を使用して [MNIST](http://yann.lecun.com/exdb/mnist/) に対するシンプルな線形画像モデルを実装する方法を説明します。

## 学習目標
1. 画像データの読み込みと表示方法を理解する
2. 誤った予測を見つけてモデルを分析する方法を理解する
3. コンピュータが画像をどのように認識しているかを視覚的に確認する

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras import Sequential
from keras.callbacks import ModelCheckpoint, TensorBoard
from keras.layers import Dense, Dropout, Flatten, Softmax

## データの探索

MNIST データセットはすでに Keras の datasets モジュールに含まれています。データを読み込んで確認してみましょう。

In [ ]:
mnist = keras.datasets.mnist.load_data()
(x_train, y_train), (x_test, y_test) = mnist

In [ ]:
HEIGHT, WIDTH = x_train[0].shape
NCLASSES = 10  # [0-9]
print("Image height x width is", HEIGHT, "x", WIDTH)
print("There are", NCLASSES, "classes")

各画像は 28 x 28 ピクセルで、0 から 9 までの数字を表しています。これらの画像は白黒（グレースケール）であるため、各ピクセルは 0（白）から 255（黒）までの値をとります。生の数値のままでは解釈が難しいことがあるため、値をプロットして手書き数字を画像として確認してみましょう。

In [ ]:
IMGNO = 12
# 生の数値を確認するにはコメントを解除してください。
# print(x_test[IMGNO])
plt.imshow(x_test[IMGNO].reshape(HEIGHT, WIDTH))
print("The label for image number", IMGNO, "is", y_test[IMGNO])

## モデルの定義
まずは非常にシンプルな線形分類器から始めましょう。これは 1998 年に MNIST で最初に試された手法で、88% の精度を記録しました。当時としては非常に画期的でした！

[Keras Sequential API](https://keras.io/api/models/sequential/) を使用して線形分類器を構築できるため、重みやバイアスを手動で定義・初期化する必要はありません。これらはバックグラウンドで自動的に処理されます。また、ロジットを確率に変換するために Softmax レイヤーを追加することもできます。最後に、誤った予測に対して高い確率を割り当てた場合に強いペナルティを与えるよう、カテゴリカル交差エントロピー（categorical cross entropy）を使用してモデルをコンパイルします。

ベンチマークとしてテストできるよう、モデルを構築して動かしてみましょう。

In [ ]:
def linear_model():
    # TODO: Sequential モデルを構築してコンパイルしてください。
    return model

## 入力関数の作成

通常どおり、トレーニングと評価のための入力関数を指定する必要があります。データの正規化の一環として、各ピクセル値を 0 から 1 の間の小数値になるようにスケーリングします。

**TODO 1**: 以下の scale 関数を定義し、データセットを構築する

In [ ]:
BUFFER_SIZE = 5000
BATCH_SIZE = 100


def scale(image, label):
    # TODO


def load_dataset(training=True):
    """MNIST データセットを tf.data.Dataset に読み込みます"""
    (x_train, y_train), (x_test, y_test) = mnist
    x = x_train if training else x_test
    y = y_train if training else y_test
    # TODO: a) ラベルをワンホットエンコーディングし、`scale` 関数を適用して、データセットを作成してください。
    # クラスをワンホットエンコーディング
    if training:
         # TODO
    return dataset

In [ ]:
def create_shape_test(training):
    dataset = load_dataset(training=training)
    data_iter = dataset.__iter__()
    (images, labels) = data_iter.get_next()
    expected_image_shape = (BATCH_SIZE, HEIGHT, WIDTH)
    expected_label_ndim = 2
    assert images.shape == expected_image_shape
    assert labels.numpy().ndim == expected_label_ndim
    test_name = "training" if training else "eval"
    print("Test for", test_name, "passed!")


create_shape_test(True)
create_shape_test(False)

いよいよモデルのトレーニングです！オリジナルの MNIST 線形分類器のエラー率は 12% でした。モデルが正常に学習しているかサニティチェックするために、このエラー率を基準として使用しましょう。

In [ ]:
NUM_EPOCHS = 10
STEPS_PER_EPOCH = 100

model = linear_model()
train_data = load_dataset()
validation_data = load_dataset(training=False)

OUTDIR = "mnist_linear"
checkpoint_callback = ModelCheckpoint(
    f"{OUTDIR}/checkpoint.weights.h5", save_weights_only=True, verbose=1
)
tensorboard_callback = TensorBoard(log_dir=OUTDIR)

history = model.fit(
    # TODO: 訓練/評価データ、エポック数、エポックあたりのステップ数を指定してください。
    callbacks=[checkpoint_callback, tensorboard_callback],
)

In [ ]:
BENCHMARK_ERROR = 0.12
BENCHMARK_ACCURACY = 1 - BENCHMARK_ERROR

accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

assert accuracy[-1] > BENCHMARK_ACCURACY
assert val_accuracy[-1] > BENCHMARK_ACCURACY
print("Test to beat benchmark accuracy passed!")

assert accuracy[0] < accuracy[1]
assert accuracy[1] < accuracy[-1]
assert val_accuracy[0] < val_accuracy[1]
assert val_accuracy[1] < val_accuracy[-1]
print("Test model accuracy is improving passed!")

assert loss[0] > loss[1]
assert loss[1] > loss[-1]
assert val_loss[0] > val_loss[1]
assert val_loss[1] > val_loss[-1]
print("Test loss is decreasing passed!")

## 予測の評価

90% 以上の精度を達成できましたか？線形推定器としては悪くない結果です！実際に予測を行い、モデルがどのような画像で間違いやすいか確認してみましょう。以下の値の範囲を変更して誤った予測を見つけ、対応する画像をプロットしてください。あなたならこれらの画像をどのように予測したでしょうか？

**TODO 2**: 以下の範囲を変更して誤った予測を見つける

In [ ]:
image_numbers = range(0, 10, 1)  # ここを変更してください。


def load_prediction_dataset():
    dataset = (x_test[image_numbers], y_test[image_numbers])
    dataset = tf.data.Dataset.from_tensor_slices(dataset)
    dataset = dataset.map(scale).batch(len(image_numbers))
    return dataset


predicted_results = model.predict(load_prediction_dataset())
for index, prediction in enumerate(predicted_results):
    predicted_value = np.argmax(prediction)
    actual_value = y_test[image_numbers[index]]
    if actual_value != predicted_value:
        print("image number: " + str(image_numbers[index]))
        print("the prediction was " + str(predicted_value))
        print("the actual label is " + str(actual_value))
        print("")

In [ ]:
bad_image_number = 8
plt.imshow(x_test[bad_image_number].reshape(HEIGHT, WIDTH));

コンピュータが苦戦するのも無理はありません。これらの画像の中には、人間にとっても判読が難しいものがあります。実は、コンピュータが各数字をどのように捉えているかを視覚化して確認することができます。

モデルの Dense レイヤーにある 10 個の各ニューロンには、785 個の重みが入力されています。これは、画像の各ピクセルに対する 1 つの重みと、1 つのバイアス項によるものです（28 × 28 + 1 = 785）。これらの重みはモデルへの入力時に平坦化（flatten）されていますが、元の画像サイズにリシェイプすることで、コンピュータが「見ている」ものを確認できます。

**TODO 3**: レイヤーの重みを入力画像の形状にリシェイプしてプロットする

In [ ]:
DIGIT = 0  # 0 から 9 までの整数に変更してください。
LAYER = 1  # レイヤー 0 は画像を平坦化するため、重みはありません
WEIGHT_TYPE = 0  # 0 は変数の重み、1 はバイアス

dense_layer_weights = model.layers[LAYER].get_weights()
digit_weights = dense_layer_weights[WEIGHT_TYPE][:, DIGIT]
plt.imshow(digit_weights.reshape((HEIGHT, WIDTH)))

コンピュータが学習しようとしていた数字がわかりましたか？なんとも不思議な画像ですね！このような単純な「脳」であっても、コンピュータは数字がどのようなものであるべきかの概念を形成できます。しかし、人間の脳は[何層もの計算を重ねて画像認識を行っています](https://www.salk.edu/news-release/brain-recognizes-eye-sees/)。

## 線形モデルからディープモデルへの拡張
シンプルな線形モデルを DNN（ディープニューラルネットワーク）モデルへと簡単に拡張できます。必要なのは、隠れ層（hidden layer）を追加することだけです。

また、ディープモデルの汎化性能を高めてよりロバストにするために、Dropout レイヤーも追加しましょう。

In [ ]:
def deep_model():
    model = Sequential(
        [
            Flatten(),
            # 隠れ層を追加
            Dense(64, activation="relu"),
            Dropout(rate=0.25),
            Dense(32, activation="relu"),
            Dense(NCLASSES),
            Softmax(),
        ]
    )

    model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )

    return model

In [ ]:
NUM_EPOCHS = 10
STEPS_PER_EPOCH = 100

model = deep_model()

OUTDIR = "mnist_dnn"
checkpoint_callback = ModelCheckpoint(
    f"{OUTDIR}/checkpoint.weights.h5", save_weights_only=True, verbose=1
)
tensorboard_callback = TensorBoard(log_dir=OUTDIR)

history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=NUM_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=[checkpoint_callback, tensorboard_callback],
)

隠れ層を追加することで精度が向上しました。しかし、各ピクセルと出力の間の関係が非線形になるため、ピクセルの寄与度（pixel attributions）を可視化することはもはや単純ではありません。

このような複雑なモデルの解釈に興味がある場合は、[こちらのノートブック](../../responsible_ai/explainable_ai/solutions/xai_image_saliency.ipynb) を参照して、モデルの挙動を理解するための高度な手法について学んでみてください。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.